In [ ]:
import os
import math
import random
import numpy as np
import matplotlib.pyplot as plt
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
import warnings
warnings.filterwarnings("ignore")

random.seed(42)
np.random.seed(42)

# Output folder setup
SCRIPT_DIR = os.path.dirname(os.path.abspath(__file__)) or "."
OUTPUT_DIR = os.path.join(SCRIPT_DIR, "outputs")
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("=" * 58)
print("       ANOMALY DETECTION — FRAUD DETECTION")
print("=" * 58)

In [ ]:
print("\n📊 Generating synthetic credit card transaction data...")

N_NORMAL = 400
N_FRAUD  = 40

# Normal transactions
normal_amount       = np.random.normal(loc=120,  scale=40,  size=N_NORMAL).clip(5, 500)
normal_time         = np.random.normal(loc=13,   scale=4,   size=N_NORMAL).clip(0, 23)
normal_num_txns     = np.random.normal(loc=5,    scale=2,   size=N_NORMAL).clip(1, 20)
normal_distance     = np.random.normal(loc=10,   scale=5,   size=N_NORMAL).clip(0, 50)
normal_foreign      = np.random.choice([0, 0, 0, 1], size=N_NORMAL)

# Fraudulent transactions (unusual patterns)
fraud_amount        = np.random.normal(loc=900,  scale=200, size=N_FRAUD).clip(300, 2000)
fraud_time          = np.random.choice(list(range(0, 5)) + list(range(22, 24)), size=N_FRAUD)
fraud_num_txns      = np.random.normal(loc=25,   scale=5,   size=N_FRAUD).clip(15, 50)
fraud_distance      = np.random.normal(loc=200,  scale=80,  size=N_FRAUD).clip(50, 500)
fraud_foreign       = np.random.choice([0, 1, 1, 1], size=N_FRAUD)

# Combine into arrays
X_normal = np.column_stack([normal_amount, normal_time, normal_num_txns,
                             normal_distance, normal_foreign])
X_fraud  = np.column_stack([fraud_amount,  fraud_time,  fraud_num_txns,
                             fraud_distance,  fraud_foreign])

X      = np.vstack([X_normal, X_fraud])
y_true = np.array([0] * N_NORMAL + [1] * N_FRAUD)

feature_names = ["Amount (₹)", "Hour of Day", "Num Transactions",
                 "Distance (km)", "Foreign Transaction"]

print(f"✅ Dataset created: {len(X)} transactions")
print(f"   Normal (0)  : {N_NORMAL}")
print(f"   Fraud  (1)  : {N_FRAUD}")
print(f"\n--- Feature Summary ---")
for i, name in enumerate(feature_names):
    print(f"   {name:<22}: Normal avg={X_normal[:,i].mean():.1f}  "
          f"Fraud avg={X_fraud[:,i].mean():.1f}")

In [ ]:
scaler   = StandardScaler()
X_scaled = scaler.fit_transform(X)

In [ ]:
print("\n" + "=" * 58)
print("  METHOD 1 : Z-Score (Statistical)")
print("=" * 58)

THRESHOLD = 2.5
z_scores  = np.abs(X_scaled)
z_max     = z_scores.max(axis=1)
z_pred    = (z_max > THRESHOLD).astype(int)

z_fraud_caught = int(((z_pred == 1) & (y_true == 1)).sum())
z_accuracy     = int((z_pred == y_true).sum()) / len(y_true) * 100
print(f"   Threshold  : Z > {THRESHOLD}")
print(f"   Flagged    : {z_pred.sum()} transactions")
print(f"   Fraud Caught: {z_fraud_caught} / {N_FRAUD}")
print(f"   Accuracy   : {z_accuracy:.1f}%")

In [ ]:
print("\n" + "=" * 58)
print("  METHOD 2 : IQR (Interquartile Range)")
print("=" * 58)

iqr_outlier = np.zeros(len(X), dtype=bool)
for i in range(X.shape[1]):
    col = X[:, i]
    Q1, Q3 = np.percentile(col, 25), np.percentile(col, 75)
    IQR     = Q3 - Q1
    lower, upper = Q1 - 1.5 * IQR, Q3 + 1.5 * IQR
    iqr_outlier |= (col < lower) | (col > upper)

iqr_pred         = iqr_outlier.astype(int)
iqr_fraud_caught = int(((iqr_pred == 1) & (y_true == 1)).sum())
iqr_accuracy     = int((iqr_pred == y_true).sum()) / len(y_true) * 100
print(f"   Flagged    : {iqr_pred.sum()} transactions")
print(f"   Fraud Caught: {iqr_fraud_caught} / {N_FRAUD}")
print(f"   Accuracy   : {iqr_accuracy:.1f}%")

In [ ]:
print("\n" + "=" * 58)
print("  METHOD 3 : Isolation Forest (ML-Based)")
print("=" * 58)

iso = IsolationForest(contamination=0.09, random_state=42, n_estimators=100)
iso.fit(X_scaled)
iso_raw  = iso.predict(X_scaled)
iso_pred = (iso_raw == -1).astype(int)

iso_fraud_caught = int(((iso_pred == 1) & (y_true == 1)).sum())
iso_accuracy     = int((iso_pred == y_true).sum()) / len(y_true) * 100
print(f"   Flagged    : {iso_pred.sum()} transactions")
print(f"   Fraud Caught: {iso_fraud_caught} / {N_FRAUD}")
print(f"   Accuracy   : {iso_accuracy:.1f}%")
print("\n📋 Isolation Forest Classification Report:")
print(classification_report(y_true, iso_pred,
                             target_names=["Normal", "Fraud"],
                             zero_division=0))

In [ ]:
print("\n" + "=" * 58)
print("  METHOD 4 : Local Outlier Factor (Density-Based)")
print("=" * 58)

lof      = LocalOutlierFactor(n_neighbors=20, contamination=0.09)
lof_raw  = lof.fit_predict(X_scaled)
lof_pred = (lof_raw == -1).astype(int)

lof_fraud_caught = int(((lof_pred == 1) & (y_true == 1)).sum())
lof_accuracy     = int((lof_pred == y_true).sum()) / len(y_true) * 100
print(f"   Flagged    : {lof_pred.sum()} transactions")
print(f"   Fraud Caught: {lof_fraud_caught} / {N_FRAUD}")
print(f"   Accuracy   : {lof_accuracy:.1f}%")

In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(16, 12))
fig.suptitle("Anomaly Detection — Fraud Detection Results", fontsize=15, fontweight="bold")

# Plot 1: Transaction Amount Distribution
ax = axes[0, 0]
ax.hist(X_normal[:, 0], bins=30, alpha=0.6, color="#2ecc71",
        edgecolor="white", label=f"Normal (n={N_NORMAL})")
ax.hist(X_fraud[:, 0],  bins=15, alpha=0.8, color="#e74c3c",
        edgecolor="white", label=f"Fraud (n={N_FRAUD})")
ax.set_xlabel("Transaction Amount (₹)")
ax.set_ylabel("Frequency")
ax.set_title("Transaction Amount Distribution", fontweight="bold")
ax.legend()
ax.axvline(np.mean(X_normal[:, 0]), color="#27ae60", lw=2, linestyle="--",
           label=f"Normal Mean: ₹{np.mean(X_normal[:,0]):.0f}")
ax.axvline(np.mean(X_fraud[:, 0]),  color="#c0392b", lw=2, linestyle="--",
           label=f"Fraud Mean: ₹{np.mean(X_fraud[:,0]):.0f}")

# Plot 2: Scatter — Amount vs Hour
ax = axes[0, 1]
ax.scatter(X_normal[:, 1], X_normal[:, 0], c="#2ecc71", alpha=0.5,
           s=30, label="Normal", edgecolors="none")
ax.scatter(X_fraud[:, 1],  X_fraud[:, 0],  c="#e74c3c", alpha=0.8,
           s=60, label="Fraud", marker="X", edgecolors="black", linewidths=0.5)
ax.set_xlabel("Hour of Day")
ax.set_ylabel("Transaction Amount (₹)")
ax.set_title("Amount vs Hour of Day\n(True Labels)", fontweight="bold")
ax.legend()

# Plot 3: Confusion Matrix — Isolation Forest
ax = axes[1, 0]
cm = confusion_matrix(y_true, iso_pred)
ConfusionMatrixDisplay(cm, display_labels=["Normal", "Fraud"]).plot(
    ax=ax, colorbar=False, cmap="RdYlGn"
)
ax.set_title("Confusion Matrix\nIsolation Forest", fontweight="bold")

# Plot 4: Method Comparison
ax = axes[1, 1]
methods       = ["Z-Score", "IQR", "Isolation\nForest", "Local Outlier\nFactor"]
accuracies    = [z_accuracy, iqr_accuracy, iso_accuracy, lof_accuracy]
fraud_caught  = [z_fraud_caught, iqr_fraud_caught, iso_fraud_caught, lof_fraud_caught]
bar_colors    = ["#3498db", "#9b59b6", "#e74c3c", "#f39c12"]

x_pos = np.arange(len(methods))
width = 0.38
bars1 = ax.bar(x_pos - width/2, accuracies,   width, color=bar_colors,
               alpha=0.85, edgecolor="white", label="Accuracy (%)")
bars2 = ax.bar(x_pos + width/2, [f/N_FRAUD*100 for f in fraud_caught],
               width, color=bar_colors, alpha=0.4, edgecolor="white",
               label="Fraud Recall (%)", hatch="//")

ax.set_xticks(x_pos)
ax.set_xticklabels(methods, fontsize=9)
ax.set_ylabel("Score (%)")
ax.set_ylim(0, 115)
ax.set_title("Method Comparison\nAccuracy vs Fraud Recall", fontweight="bold")
ax.legend(fontsize=8)
for bar, val in zip(bars1, accuracies):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{val:.0f}%", ha="center", fontsize=8, fontweight="bold")
for bar, val in zip(bars2, [f/N_FRAUD*100 for f in fraud_caught]):
    ax.text(bar.get_x() + bar.get_width()/2, bar.get_height() + 1,
            f"{val:.0f}%", ha="center", fontsize=8, color="#555")

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR, "anomaly_detection_results.png"),
            dpi=150, bbox_inches="tight")
plt.show()
print("✅ Plots saved.")

In [ ]:
print("\n" + "=" * 58)
print("           LIVE TRANSACTION CHECKER")
print("=" * 58)

def check_transaction(amount, hour, num_txns, distance, is_foreign):
    """Check if a transaction is fraudulent using Isolation Forest."""
    txn    = np.array([[amount, hour, num_txns, distance, is_foreign]])
    scaled = scaler.transform(txn)
    pred   = iso.predict(scaled)[0]
    result = "🚨 FRAUD ALERT" if pred == -1 else "✅ NORMAL"
    score  = iso.decision_function(scaled)[0]
    print(f"\n   Amount=₹{amount:<6}  Hour={hour}:00  Txns={num_txns}"
          f"  Dist={distance}km  Foreign={'Yes' if is_foreign else 'No'}")
    print(f"   → {result}  (anomaly score: {score:.3f})")

print("\n Transaction 1 — Small morning purchase:")
check_transaction(amount=85,   hour=10, num_txns=3,  distance=5,   is_foreign=0)

print("\n Transaction 2 — Large late-night foreign transaction:")
check_transaction(amount=1500, hour=2,  num_txns=30, distance=350, is_foreign=1)

print("\n Transaction 3 — Medium evening local purchase:")
check_transaction(amount=200,  hour=18, num_txns=6,  distance=12,  is_foreign=0)

print("\n Transaction 4 — Unusual burst of transactions:")
check_transaction(amount=750,  hour=3,  num_txns=40, distance=180, is_foreign=1)

In [ ]:
print("\n" + "=" * 58)
print("               FINAL SUMMARY")
print("=" * 58)
print(f"  {'Method':<22} {'Accuracy':>9} {'Fraud Caught':>13}")
print("  " + "-" * 46)
methods_clean = ["Z-Score", "IQR", "Isolation Forest", "Local Outlier Factor"]
for name, acc, caught in zip(methods_clean,
                              [z_accuracy, iqr_accuracy, iso_accuracy, lof_accuracy],
                              [z_fraud_caught, iqr_fraud_caught, iso_fraud_caught, lof_fraud_caught]):
    print(f"  {name:<22} {acc:>8.1f}%  {caught:>5}/{N_FRAUD} fraud txns")

best = methods_clean[np.argmax([z_accuracy, iqr_accuracy, iso_accuracy, lof_accuracy])]
print(f"\n🏆 Best Method : {best}")
print("✅ Experiment complete!")